<a href="https://colab.research.google.com/github/yosetcruz/Redes-Neuronales/blob/main/RedesNeuronalesT3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import tensorflow as tf 
from tensorflow import keras 
from tensorflow.keras import layers
from tensorflow.keras import regularizers 
import optuna 
import wandb  


c:\Users\yoset cruz\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
wandb.login(key="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")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\yoset cruz\_netrc
wandb: Currently logged in as: josephcruz7321 (josephcruz7321-benemerita-universidad-autonoma-de-puebla) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
(x_train, y_train), (x_test, y_test)= keras.datasets.mnist.load_data()

# normalizar
x_train = x_train / 255.0
x_test = x_test / 255.0

# aplanar imagenes
x_train = x_train.reshape(-1, 784)
x_test = x_test.reshape(-1, 784)

print(x_train.shape)

(60000, 784)


In [6]:
def create_model(trial):

    n_layers = trial.suggest_int("n_layers", 1, 3)

    model = keras.Sequential()
    model.add(layers.Input(shape=(784,)))

    for i in range(n_layers):

        units = trial.suggest_int(f"units_{i}", 32, 256)

        activation = trial.suggest_categorical(
            f"activation_{i}",
            ["relu", "tanh"]
        )

        model.add(layers.Dense(units, activation=activation))

    model.add(layers.Dense(10, activation="softmax"))

    optimizer_name = trial.suggest_categorical(
        "optimizer",
        ["adam", "rmsprop"]
    )

    model.compile(
        optimizer=optimizer_name,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [7]:
def objective(trial):

    model = create_model(trial)

    history = model.fit(
        x_train,
        y_train,
        epochs=5,
        batch_size=128,
        validation_split=0.2,
        verbose=0
    )

    loss, accuracy = model.evaluate(x_test, y_test, verbose=0)

    return accuracy

In [8]:
study = optuna.create_study(direction="maximize")

study.optimize(objective, n_trials=10)

print("Best accuracy:", study.best_value)
print("Best params:", study.best_params)

[I 2026-03-06 22:34:08,466] A new study created in memory with name: no-name-3103c0a5-7142-467f-a532-914cf337a20e


[I 2026-03-06 22:34:45,368] Trial 0 finished with value: 0.9707000255584717 and parameters: {'n_layers': 1, 'units_0': 251, 'activation_0': 'tanh', 'optimizer': 'rmsprop'}. Best is trial 0 with value: 0.9707000255584717.
[I 2026-03-06 22:35:12,657] Trial 1 finished with value: 0.9749000072479248 and parameters: {'n_layers': 3, 'units_0': 196, 'activation_0': 'relu', 'units_1': 245, 'activation_1': 'tanh', 'units_2': 230, 'activation_2': 'relu', 'optimizer': 'adam'}. Best is trial 1 with value: 0.9749000072479248.
[I 2026-03-06 22:35:28,358] Trial 2 finished with value: 0.9700999855995178 and parameters: {'n_layers': 3, 'units_0': 87, 'activation_0': 'tanh', 'units_1': 86, 'activation_1': 'relu', 'units_2': 121, 'activation_2': 'relu', 'optimizer': 'adam'}. Best is trial 1 with value: 0.9749000072479248.
[I 2026-03-06 22:35:43,644] Trial 3 finished with value: 0.9649999737739563 and parameters: {'n_layers': 1, 'units_0': 156, 'activation_0': 'tanh', 'optimizer': 'adam'}. Best is trial 1

Best accuracy: 0.9749000072479248
Best params: {'n_layers': 3, 'units_0': 196, 'activation_0': 'relu', 'units_1': 245, 'activation_1': 'tanh', 'units_2': 230, 'activation_2': 'relu', 'optimizer': 'adam'}


In [9]:
best_params = study.best_params

model = keras.Sequential()
model.add(layers.Input(shape=(784,)))

n_layers = best_params["n_layers"]

for i in range(n_layers):

    units = best_params[f"units_{i}"]
    activation = best_params[f"activation_{i}"]

    model.add(layers.Dense(units, activation=activation))

model.add(layers.Dense(10, activation="softmax"))

model.compile(
    optimizer=best_params["optimizer"],
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [10]:
history = model.fit(
    x_train,
    y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.2
)

Epoch 1/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.9160 - loss: 0.2786 - val_accuracy: 0.9585 - val_loss: 0.1383
Epoch 2/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.9683 - loss: 0.1043 - val_accuracy: 0.9681 - val_loss: 0.1045
Epoch 3/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.9771 - loss: 0.0720 - val_accuracy: 0.9722 - val_loss: 0.0918
Epoch 4/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9836 - loss: 0.0506 - val_accuracy: 0.9685 - val_loss: 0.1064
Epoch 5/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9867 - loss: 0.0414 - val_accuracy: 0.9673 - val_loss: 0.1154
Epoch 6/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9897 - loss: 0.0323 - val_accuracy: 0.9739 - val_loss: 0.0998
Epoch 7/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9901 - loss: 0.0290 - val_accuracy: 0.9752 - val_loss: 0.1017
Epoch 8/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.9917 - loss: 0.0247 - val_accu

In [12]:
loss, accuracy = model.evaluate(x_test, y_test)
print("Test accuracy:", accuracy)

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9755 - loss: 0.1014
Test accuracy: 0.9754999876022339


In [14]:
model_reg = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(
        128,
        activation="relu",
        kernel_regularizer=regularizers.l2(0.001)
    ),
    layers.Dense(10, activation="softmax")
])

model_reg.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model_reg.fit(
    x_train,
    y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.2
)

Epoch 1/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 28s 6ms/step - accuracy: 0.8915 - loss: 0.5283 - val_accuracy: 0.9408 - val_loss: 0.3313
Epoch 2/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.9417 - loss: 0.3023 - val_accuracy: 0.9538 - val_loss: 0.2633
Epoch 3/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.9541 - loss: 0.2463 - val_accuracy: 0.9608 - val_loss: 0.2244
Epoch 4/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9620 - loss: 0.2146 - val_accuracy: 0.9628 - val_loss: 0.2081
Epoch 5/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9661 - loss: 0.1951 - val_accuracy: 0.9640 - val_loss: 0.1999
Epoch 6/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9685 - loss: 0.1819 - val_accuracy: 0.9687 - val_loss: 0.1833
Epoch 7/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9713 - loss: 0.1706 - val_accuracy: 0.9707 - val_loss: 0.1721
Epoch 8/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.9730 - loss: 0.1628 - val_accuracy: 

In [13]:
model_l2 = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(
        128,
        activation="relu",
        kernel_regularizer=regularizers.l2(0.001)
    ),
    layers.Dense(10, activation="softmax")
])

model_l2.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model_l2.fit(
    x_train,
    y_train,
    epochs=10,
    validation_split=0.2
)

Epoch 1/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 0.9143 - loss: 0.4166 - val_accuracy: 0.9494 - val_loss: 0.2761
Epoch 2/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 0.9538 - loss: 0.2477 - val_accuracy: 0.9610 - val_loss: 0.2208
Epoch 3/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9629 - loss: 0.2116 - val_accuracy: 0.9622 - val_loss: 0.2102
Epoch 4/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.9667 - loss: 0.1932 - val_accuracy: 0.9671 - val_loss: 0.1887
Epoch 5/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.9684 - loss: 0.1840 - val_accuracy: 0.9703 - val_loss: 0.1786
Epoch 6/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - accuracy: 0.9714 - loss: 0.1729 - val_accuracy: 0.9712 - val_loss: 0.1757
Epoch 7/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - accuracy: 0.9711 - loss: 0.1694 - val_accuracy: 0.9704 - val_loss: 0.1738
Epoch 8/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.9728 - loss: 0.162

In [14]:
model_dropout = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(10, activation="softmax")
])

model_dropout.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model_dropout.fit(
    x_train,
    y_train,
    epochs=10,
    validation_split=0.2
)

Epoch 1/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.8977 - loss: 0.3477 - val_accuracy: 0.9533 - val_loss: 0.1604
Epoch 2/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9481 - loss: 0.1773 - val_accuracy: 0.9655 - val_loss: 0.1180
Epoch 3/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.9575 - loss: 0.1364 - val_accuracy: 0.9700 - val_loss: 0.1044
Epoch 4/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.9654 - loss: 0.1134 - val_accuracy: 0.9727 - val_loss: 0.0924
Epoch 5/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.9691 - loss: 0.0999 - val_accuracy: 0.9752 - val_loss: 0.0852
Epoch 6/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - accuracy: 0.9728 - loss: 0.0870 - val_accuracy: 0.9750 - val_loss: 0.0813
Epoch 7/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.9746 - loss: 0.0797 - val_accuracy: 0.9752 - val_loss: 0.0870
Epoch 8/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.9767 - loss: 0.0718

In [15]:
import wandb
wandb.init(project="prueba")
for i in range(10):
    wandb.log({"accuracy": i/10})
wandb.finish()

wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


accuracy,▁▂▃▃▄▅▆▆▇█
accuracy,0.9
